# Pipeline completa: dados, circuitos, transpilacao e navegacao

Este notebook mostra um fluxo completo e executavel da biblioteca:

1. entrada de dados classicos;
2. construcao do circuito de navegacao V1;
3. circuito de consulta que prepara um endereco e mede o registrador de saida;
4. pipeline enriquecida com snapshots, stages, transformacoes e circuito transpilado;
5. execucao local quando `qiskit-aer` estiver disponivel;
6. Navigation V2 sobre heap estrutural tipada;
7. lowering V2 -> V1/CircuitIR -> Qiskit;
8. recuperacao do dado navegado por counts canonicos.

Nota importante: `PipelineResult` nao e um circuito. Para visualizar uma pipeline, use `show_circuits()`, `show_transformations()`, `show_measurements()` ou os snapshots `before_transpile` e `after_transpile`.

In [ ]:
from quantum_cq import (
    CQ,
    StructuralField,
    StructuralHeap,
    StructuralNode,
    StructuralSelector,
    StructuralType,
)
from qiskit import QuantumCircuit

try:
    from IPython.display import display
except Exception:
    display = None

print("quantum-cq", __import__("quantum_cq").__version__)

## 1. Helpers de visualizacao da pipeline

As funcoes abaixo mostram a pipeline pelo contrato publico atual: scenario, stages, snapshots, grafo de transformacoes e counts canonicos. Elas evitam chamar `CQ.show(...)` no `PipelineResult` inteiro.

In [ ]:
def _metric_payload(metrics):
    payload = {}
    for key, value in dict(metrics or {}).items():
        payload[key] = getattr(value, "value", value)
    return payload


def ir_operations(ir):
    rows = []
    for layer_index, layer in enumerate(getattr(ir, "layers", ())):
        for op in getattr(layer, "operations", ()): 
            rows.append({
                "where": f"layer-{layer_index}",
                "kind": op.kind,
                "qubits": tuple(op.qubits),
                "clbits": tuple(op.clbits),
                "params": dict(op.params),
            })
    for op in getattr(ir, "outputs", ()): 
        rows.append({
            "where": "outputs",
            "kind": op.kind,
            "qubits": tuple(op.qubits),
            "clbits": tuple(op.clbits),
            "params": dict(op.params),
        })
    return rows


def show_qiskit_circuit(title, circuit):
    print(f"\n=== {title} ===")
    print({
        "type": type(circuit).__name__,
        "qubits": getattr(circuit, "num_qubits", None),
        "clbits": getattr(circuit, "num_clbits", None),
        "depth": circuit.depth() if hasattr(circuit, "depth") else None,
        "ops": dict(circuit.count_ops()) if hasattr(circuit, "count_ops") else None,
    })
    if display is not None and hasattr(circuit, "draw"):
        try:
            display(circuit.draw("text"))
        except Exception as exc:
            print("desenho textual indisponivel:", type(exc).__name__, exc)


def show_ir(title, ir):
    print(f"\n=== {title} ===")
    print({
        "type": type(ir).__name__,
        "name": getattr(ir, "name", None),
        "qubits": getattr(ir, "n_qubits", None),
        "clbits": getattr(ir, "n_clbits", None),
        "operations": ir_operations(ir),
    })
    try:
        show_qiskit_circuit(title + " emitido para Qiskit", CQ.emit(ir, engine="qiskit"))
    except Exception as exc:
        print("emissao para Qiskit indisponivel:", type(exc).__name__, exc)


def explain_pipeline(result, title="pipeline"):
    print(f"\n### {title}")
    scenario = result.scenario_results[0]
    print("scenario_id:", scenario.scenario_id)
    print("status:", scenario.status)
    print("terminal artifacts:", sorted(scenario.artifacts.keys()))

    print("\nStages:")
    for stage in scenario.stage_results:
        print({
            "stage": stage.stage_id,
            "status": stage.status,
            "requires": tuple(stage.requires),
            "provides": tuple(stage.provides),
            "diagnostics": tuple(stage.diagnostics),
        })

    print("\nSnapshots:")
    for snapshot in scenario.snapshots:
        print({
            "snapshot": snapshot.snapshot_id,
            "stage": snapshot.stage_id,
            "format": snapshot.format,
            "engine": snapshot.engine,
            "target": snapshot.target,
            "metrics": _metric_payload(snapshot.metrics),
        })

    print("\nTransformacoes:")
    result.show_transformations()
    return scenario


def show_pipeline_circuits(result):
    scenario = result.scenario_results[0]
    for snapshot in scenario.snapshots:
        label = f"{snapshot.stage_id} / {snapshot.format}"
        circuit = snapshot.circuit
        if snapshot.format == "ir":
            show_ir(label, circuit)
        elif hasattr(circuit, "draw"):
            show_qiskit_circuit(label, circuit)
        else:
            print(label, type(circuit).__name__)


def decode_counts_as_int(counts):
    bitstring = max(counts, key=counts.get)
    return bitstring, int(bitstring, 2) if bitstring else 0

## 2. Entrada dos dados

Vamos usar uma memoria classica simples. O objetivo e consultar o endereco `2` e recuperar o valor `7`.

In [ ]:
values = [3, 5, 7, 9]
address = 2
expected = values[address]
shots = 128

print("dados:", values)
print("endereco consultado:", address)
print("valor esperado:", expected)

## 3. Navigation V1: operador de memoria enderecada

A V1 constr?i o operador reversivel `|a>|b> -> |a>|b xor D[a]>`. Ele nao mede nem prepara o endereco sozinho; isso fica no circuito de consulta.

In [ ]:
nav_v1 = CQ.nav(values, engine="explicit")
nav_v1_qiskit = CQ.to_qiskit(nav_v1)

print("metadata V1:")
for key in ("navigation_name", "engine", "address_qubits", "data_qubits", "access_semantics", "physical_qram"):
    print(f"  {key}:", nav_v1.metadata[key])

show_qiskit_circuit("Operador Navigation V1", nav_v1_qiskit)

## 4. Circuito de consulta V1

Agora criamos um circuito que:

1. prepara o endereco `2` no registrador de endereco;
2. aplica o operador `CQ.nav(...)`;
3. mede apenas os qubits de dados.

Os counts canonicos usam clbits em ordem descendente, entao a string `0111` corresponde ao inteiro `7`.

In [ ]:
def build_v1_read_circuit(values, address):
    nav = CQ.nav(values, engine="explicit")
    nav_qc = CQ.to_qiskit(nav)
    address_qubits = nav.metadata["address_qubits"]
    data_qubits = nav.metadata["data_qubits"]
    total_qubits = address_qubits + data_qubits

    query = QuantumCircuit(total_qubits, data_qubits, name=f"v1_read_address_{address}")
    for bit in range(address_qubits):
        if (address >> bit) & 1:
            query.x(bit)
    query.compose(nav_qc, inplace=True)
    for bit in range(data_qubits):
        query.measure(address_qubits + bit, bit)
    return query, nav

v1_query, _ = build_v1_read_circuit(values, address)
show_qiskit_circuit("Circuito de consulta V1", v1_query)

## 5. Pipeline: transpilacao e processo completo do circuito V1

Aqui a entrada primaria da pipeline e o circuito de consulta. A pipeline registra stages, snapshots, transformacoes e os circuitos antes/depois da transpilacao nativa.

In [ ]:
transpile_builder = CQ.pipeline(circuit=v1_query, engine="qiskit")
try:
    from qiskit_aer import AerSimulator
    # Backend local explicito para a etapa de transpilacao nativa.
    transpile_builder = transpile_builder.with_runtime(backend=AerSimulator(), optimization_level=1)
except Exception as exc:
    print("qiskit-aer indisponivel para backend local explicito:", type(exc).__name__, exc)

v1_transpiled = transpile_builder.transpile()
scenario = explain_pipeline(v1_transpiled, "V1 read: transpile")
show_pipeline_circuits(v1_transpiled)

print("\nTranspilation record:")
print(v1_transpiled.transpilation_record)

## 6. Compilacao e execucao V1

A execucao local usa `qiskit-aer`. Se Aer nao estiver instalado, a celula registra o motivo e segue sem quebrar o notebook.

In [ ]:
v1_compiled = CQ.pipeline(circuit=v1_query, engine="qiskit").compile(engine="qiskit")
print("compiled_artifact.engine:", v1_compiled.compiled_artifact.engine)
print("compiled measurement contract:", v1_compiled.compiled_artifact.measurement_contract)

try:
    v1_executed = CQ.pipeline(circuit=v1_query, engine="qiskit").run_engine(engine="qiskit", shots=shots)
    explain_pipeline(v1_executed, "V1 read: run_engine")
    v1_executed.show_measurements()
    bitstring, recovered = decode_counts_as_int(v1_executed.engine_result.counts)
    print("bitstring dominante:", bitstring)
    print("valor recuperado:", recovered)
    assert recovered == expected
except Exception as exc:
    v1_executed = None
    print("execucao V1 pulada:", type(exc).__name__, exc)

## 7. Navigation V2: heap estrutural tipada

Agora modelamos os mesmos dados como uma lista ligada finita. A V2 valida a estrutura, canonicaliza por renomeacao, planeja uma representacao finita e baixa a operacao para o CircuitIR atual usando o backend V1 como mecanismo concreto.

In [ ]:
node_type = StructuralType(
    "Node",
    (
        StructuralField("payload", "uint", bit_width=4, semantic_role="value"),
        StructuralField("link", "reference", nullable=True, semantic_role="next"),
    ),
)
heap = StructuralHeap(
    (node_type,),
    (
        StructuralNode("node0", "Node", {"payload": values[0], "link": "node1"}),
        StructuralNode("node1", "Node", {"payload": values[1], "link": "node2"}),
        StructuralNode("node2", "Node", {"payload": values[2], "link": "node3"}),
        StructuralNode("node3", "Node", {"payload": values[3], "link": None}),
    ),
    roots=("node0",),
)

nav_v2 = CQ.navigation_v2(heap, operation="read", selector=StructuralSelector.value("payload"))

print("navigation version:", nav_v2.navigation_version)
print("operation:", nav_v2.operation.operation)
print("circuit format:", nav_v2.circuit_format)
print("equivalence fingerprint:", nav_v2.plan.equivalence_class.equivalence_fingerprint)
print("local -> canonical:", dict(nav_v2.equivalence_class.local_to_canonical))
print("pointer codes:", dict(nav_v2.plan.register_layout["pointer"]["codes"]))
print("rho/memory table:", nav_v2.plan.memory_values)
print("lowering metadata:", nav_v2.metadata)

show_ir("CircuitIR baixado da Navigation V2", nav_v2.circuit)

## 8. Pipeline estrutural V2

Neste terminal, a entrada primaria e `structural_navigation=nav_v2`. A pipeline executa os stages estruturais antes de seguir para CircuitIR e transpilacao.

In [ ]:
v2_structural_pipeline = CQ.pipeline(structural_navigation=nav_v2).transpile()
explain_pipeline(v2_structural_pipeline, "V2 structural navigation: transpile")
show_pipeline_circuits(v2_structural_pipeline)

## 9. Navegando na V2 e recuperando o dado

Para recuperar um valor especifico, preparamos o ponteiro canonicalizado de `node2`, aplicamos o circuito baixado da V2 e medimos o acumulador de saida.

In [ ]:
def build_v2_read_circuit(nav_result, local_node_id):
    canonical_id = nav_result.equivalence_class.local_to_canonical[local_node_id]
    pointer_code = nav_result.plan.register_layout["pointer"]["codes"][canonical_id]
    pointer_width = nav_result.plan.pointer_width
    output_width = nav_result.plan.output_width
    operator_qc = CQ.to_qiskit(nav_result.circuit)

    query = QuantumCircuit(pointer_width + output_width, output_width, name=f"v2_read_{local_node_id}")
    for bit in range(pointer_width):
        if (pointer_code >> bit) & 1:
            query.x(bit)
    query.compose(operator_qc, inplace=True)
    for bit in range(output_width):
        query.measure(pointer_width + bit, bit)
    return query, canonical_id, pointer_code

v2_query, canonical_id, pointer_code = build_v2_read_circuit(nav_v2, "node2")
print("local node:", "node2")
print("canonical node:", canonical_id)
print("encoded pointer value:", pointer_code)
show_qiskit_circuit("Circuito de consulta V2 para node2", v2_query)

In [ ]:
v2_transpiled = CQ.pipeline(circuit=v2_query, engine="qiskit").transpile()
explain_pipeline(v2_transpiled, "V2 query circuit: transpile")
show_pipeline_circuits(v2_transpiled)

try:
    v2_executed = CQ.pipeline(circuit=v2_query, engine="qiskit").run_engine(engine="qiskit", shots=shots)
    explain_pipeline(v2_executed, "V2 query circuit: run_engine")
    v2_executed.show_measurements()
    bitstring, recovered = decode_counts_as_int(v2_executed.engine_result.counts)
    print("bitstring dominante:", bitstring)
    print("valor recuperado:", recovered)
    assert recovered == expected
except Exception as exc:
    v2_executed = None
    print("execucao V2 pulada:", type(exc).__name__, exc)

## 10. Comparacao final

O ponto principal e que V1 e V2 coexistem:

- V1: memoria enderecada concreta e estavel;
- V2: camada estrutural exata, canonicalizada e baixada para o mecanismo concreto atual;
- ambas recuperam o mesmo valor observavel para o exemplo finito.

In [ ]:
summary = {
    "dados": values,
    "consulta": {"address": address, "local_node": "node2"},
    "esperado": expected,
    "v1_counts": None if v1_executed is None else dict(v1_executed.engine_result.counts),
    "v2_counts": None if v2_executed is None else dict(v2_executed.engine_result.counts),
    "v1_default_api_preservada": CQ.available_navigation_encodings(),
    "v2_api_explicita": nav_v2.navigation_version,
}
summary